In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import chi2_contingency

In [ ]:
df = pd.read_csv("../data/depression_data.csv")

## The Basics

In [ ]:
# Display some basic info of the df like number of rows, columns
# the dataframe head, and descriptive info of numerical columns
print(df.shape)
display(df.head())
display(df.describe().T)

A small selection of features here with only a few numerical columns. I'll be treating this as a classification problem as the "history of mental illness" that I want to predict is a yes/no column.   
The info from the kaggle site on the dataset suggested that this data was already relatively clean and deduped but worth checking for safety:

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
# Inspect the target counts
df["History of Mental Illness"].value_counts(dropna=False)

In [ ]:
# Show target graphically - predominantly "no"
sns.countplot(data=df, x="History of Mental Illness")

In [ ]:
# Check the proportions of yes / no in the target
df["History of Mental Illness"].value_counts(normalize=True)

## Numeric Columns

In [ ]:
# Select the numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

In [ ]:
# Plot histograms of each of the numeric features
df[numeric_cols].hist(figsize=(15,12))

In [ ]:
# Print out table of statistical values
print(
    f"{'Feature':<18} | {'Mean':<8} | "
    f"{'Median':<8} | {'Standard Deviation':<8}"
)
for column in numeric_cols:
    print(
        f"{column:<18} | {df[column].mean():<8.2f} | "
        f"{df[column].median():<8.2f} | {df[column].std():<8.2f}"
    )

##### Visually inspect numerical feature distributions

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Age")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Number of Children")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Income")

Some small differences in age, very similar distributions by number of children, very different shapes when considering income. 
To me this suggests that there is some correlation with income, where higher income customers may be less likely to havea a history of mental illness.  
This is not necessarily causal (e.g. a low income job may be unlikely to result in mental illness, but those without mental illnesses may find it easier to secure a higher salary position)

## Categorical Columns

In [ ]:
# Select categorical columns
categorical_cols = list(set(df.columns)-set(numeric_cols))
categorical_cols.remove("Name")

In [ ]:
print(categorical_cols)

##### Visually inspect categorical feature distributions

In [ ]:
pd.crosstab(
    df["History of Substance Abuse"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Marital Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Education Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Family History of Depression"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Alcohol Consumption"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Smoking Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Physical Activity Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Chronic Medical Conditions"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Sleep Patterns"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Dietary Habits"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Employment Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

These distributions suggest some small correlation between a history of mental health issues and:
 * Marital status
 * Alcohol consumption   

and larger correlation between a history of mental health issues and:
 * Sleep patterns 
 * Dietary habits
 * Employment status

## Statistical Tests

In [ ]:
for column in categorical_cols:
    table = pd.crosstab(
        df[column],
        df["History of Mental Illness"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    print(f"Column {column}, p: {p:.2f}")

The p values of each categorical feature suggest that the relationship between the target and the `History of Substance Abuse` feature is considerably more likely than the other features to occur through random chance.   
Considering the typical p value limit is around 0.05, I will likely not include this column in my model. 
Additionally, while there are no issues with the p value of the `Chronic Medical Conditions` column I have reservations against using it in the model: it's correlation to the target seems minimal, and in a real world scenario I would worry about the potential bias that may arise based on the nature of these conditions.